# P6 — Wi-Fi Indoor Localisation
**Group G6 · Paracetamol** — Applied Machine Learning Lab

| Member | Roll no. | Owns (model) | Owns (notebook / report section) |
|---|---|---|---|
| Philem Arjun | 2481034 | K-Means | §0 Setup & split, §1 EDA, §2 Unsupervised clustering |
| Dhruv Rangan | 2481048 | KNN, SVM | §4 Supervised methodology (pipeline, metric, tuning) |
| Yashveer Singh | 2481242 | Random Forest | §5 Robustness (AP failure, noise), §6 Final test |
| Harsh Kumar | 2481121 | Logistic Regression (+ Dummy baseline), Decision Tree | §3 PCA, §7 Interpretation & limitations |

**Problem.** Predict which of 4 rooms a smartphone is in from the received signal strength (RSSI, in dBm) of 7 Wi-Fi access points (APs) — first without labels (clustering), then with labels (classification).

**Dataset.** UCI *Wireless Indoor Localization* (Rohra et al., 2017; CC BY 4.0): 2,000 readings × 7 APs, label = room 1–4.

**How to run.** `Kernel → Restart & Run All`. Every random step uses `RANDOM_STATE = 42`, so the outputs are identical on every run.

**Notebook map:** 0 Setup → split → 1 EDA → 2 K-Means → 3 PCA → 4 Supervised models → 5 Robustness (CV) → 6 Final test-set evaluation (test set used **once**) → 7 Interpretation → 8 Limitations

## 0. Setup and reproducibility

In [1]:
import sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from scipy.optimize import linear_sum_assignment

from sklearn.base import clone
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_validate)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.feature_selection import f_classif
from sklearn.metrics import (f1_score, accuracy_score, adjusted_rand_score, silhouette_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

# ---- one seed for everything -------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
N_JOBS = -1          # parallelism does not change results: every estimator has a fixed random_state

DATA_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/00422/wifi_localization.txt"
DATA_PATH = Path("data/wifi_localization.txt")
FIG_DIR   = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
RES_DIR   = Path("results"); RES_DIR.mkdir(exist_ok=True)

AP_COLS = [f"AP{i}" for i in range(1, 8)]
ROOMS   = [1, 2, 3, 4]
ROOM_NAMES = [f"Room {r}" for r in ROOMS]
PALETTE = dict(zip(ROOM_NAMES, sns.color_palette("tab10", 4)))

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.precision", 3)
warnings.filterwarnings("ignore", category=FutureWarning)

# 5-fold stratified CV, shuffled with the fixed seed. Used for tuning and for every mean ± SD claim.
CV       = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
CV_OUTER = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 1)  # outer loop of nested CV

def pm(scores, d=3):
    # mean ± sample standard deviation (ddof=1) across CV folds
    s = np.asarray(scores, dtype=float)
    return f"{s.mean():.{d}f} ± {s.std(ddof=1):.{d}f}"

def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn {sklearn.__version__} | matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")

Python 3.13.14 | numpy 2.2.6 | pandas 2.2.3 | scikit-learn 1.6.1 | matplotlib 3.10.3 | seaborn 0.13.2


In [2]:
# Load (downloads once from UCI if the file is not in data/)
if not DATA_PATH.exists():
    import urllib.request
    DATA_PATH.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH, sep=r"\s+", header=None, names=AP_COLS + ["room"])
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Class counts:", df["room"].value_counts().sort_index().to_dict())
df.head()

Shape: (2000, 8)
Missing values: 0
Exact duplicate rows: 0
Class counts: {1: 500, 2: 500, 3: 500, 4: 500}


,AP1,AP2,AP3,AP4,AP5,AP6,AP7,room
0,-64,-56,-61,-66,-71,-82,-81,1
1,-68,-57,-61,-65,-71,-85,-85,1
2,-63,-60,-60,-67,-76,-85,-84,1
3,-61,-60,-68,-62,-77,-90,-80,1
4,-63,-65,-60,-63,-77,-81,-87,1


### Split first (leak-free protocol)
The test set (20 %, stratified by room) is separated **before** any EDA, scaling, clustering, PCA or tuning.
Everything below — including EDA — looks only at the training set. The test set is touched exactly once, in §6.

All preprocessing (`SimpleImputer` → `StandardScaler`) lives inside a scikit-learn `Pipeline`, so inside every CV fold it is fitted on that fold's training part only.
The data has no missing values, but the imputer is there on purpose: in §5/§6 a failed access point is passed as `NaN` and the pipeline fills it with the **training** median — exactly what would happen in deployment.

In [3]:
X = df[AP_COLS]
y = df["room"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, y_train.value_counts().sort_index().to_dict())
print("Test: ", X_test.shape,  y_test.value_counts().sort_index().to_dict())

Train: (1600, 7) {1: 400, 2: 400, 3: 400, 4: 400}
Test:  (400, 7) {1: 100, 2: 100, 3: 100, 4: 100}
